In [ ]:
model = LlaDAModel()
model.enable_past_kv()

# 处理past_key_values
def run_model_semi_cache_snapshot(model, x, y, config_diffusion):

    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.sorter
    collector = config_diffusion.collector

    idx_refresh = torch.tensor([[]])


    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_blk = x[:,:position_end] == id_mask

        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

        idx_denoising = x[:, position_start:position_end]
        idx_current = torch.cat([idx_refresh, idx_denoising], dim=-1) 
        logits = model(x[:, idx_current], idx_current=idx_current).logits

        snapshot = SimpleLogitsSnapshot(logits, x, y, id_mask)
        conf_snapshot = snapshot.transform_logits(collector)

        for step in range(step_per_block):

            '''固定的重排拿一个'''
            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)    # 重全排    这里又有问题啦
            num_unmask = quota_helper.get_quota(step)
            idx_denoising = idx_sorted_by_conf[:, :num_unmask]
            idx_current = torch.cat([idx_refresh, idx_denoising], dim=-1)   # 原本的位置
            logits = model(x[:, idx_current], idx_current=idx_current).logits

            logits_denoising = logits[:, idx_denoising]

            '''固定的materilize step'''
            snapshot.update_logits_(logits_denoising, idx_denoising)
            conf_snapshot = snapshot.transform_logits(collector) # conf recal???
            snapshot.materilize_by_idx_(idx_denoising, conf_snapshot[:, idx_denoising])
            idx_refresh = idx_denoising
        # end

        p_final.scatter_(1, idx_current, snapshot.get_p_finalized())
    # end

    return torch.gather(p_final, dim=-1, index=torch.arange(len_prompt, x.shape[-1]))
# end